# 07 - Comparacion final

Este notebook calcula las metricas de backtesting y genera todas las tablas oficiales de comparacion a partir de las predicciones guardadas en `results/predictions/`.


## Metricas de backtesting

Esta celda define las metricas y tests usados para comparar todos los modelos con el mismo criterio.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2


ALPHA = 0.99
P_EXC = 1.0 - ALPHA
SIG = 0.05


def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    mask = np.isfinite(real_losses) & np.isfinite(var_preds)
    hits = (real_losses[mask] > var_preds[mask]).astype(int)
    return real_losses[mask], var_preds[mask], hits


def tick_loss(real_losses, var_preds, alpha=ALPHA):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1.0 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=ALPHA):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=ALPHA):
    _, _, hits = hit_series(real_losses, var_preds)
    n_obs = len(hits)
    n_exc = int(hits.sum())
    p = 1.0 - alpha
    if n_obs == 0 or n_exc == 0 or n_exc == n_obs:
        return np.nan
    p_hat = n_exc / n_obs
    lr_uc = -2.0 * np.log(
        ((1.0 - p) ** (n_obs - n_exc) * p**n_exc)
        / ((1.0 - p_hat) ** (n_obs - n_exc) * p_hat**n_exc)
    )
    return float(1.0 - chi2.cdf(lr_uc, df=1))


def christoffersen_cc_test(real_losses, var_preds, alpha=ALPHA):
    _, _, hits = hit_series(real_losses, var_preds)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(hits)):
        pair = (hits[t - 1], hits[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1

    if (n00 + n01) == 0 or (n10 + n11) == 0:
        return np.nan

    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11)
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l0 = ((1.0 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
    l1 = ((1.0 - pi01) ** n00) * (pi01**n01) * ((1.0 - pi11) ** n10) * (pi11**n11)
    if l0 <= 0 or l1 <= 0:
        return np.nan

    lr_ind = -2.0 * np.log(l0 / l1)
    p_uc = kupiec_test(real_losses, var_preds, alpha=alpha)
    if not np.isfinite(p_uc):
        return np.nan
    lr_uc = chi2.isf(p_uc, df=1)
    return float(1.0 - chi2.cdf(lr_uc + lr_ind, df=2))


def dq_test(real_losses, var_preds, alpha=ALPHA, lags=4):
    _, _, hits = hit_series(real_losses, var_preds)
    n_obs = len(hits)
    p = 1.0 - alpha
    if n_obs <= lags + 5:
        return np.nan

    hit = hits - p
    y = hit[lags:]
    x = np.column_stack([np.ones(len(y))] + [hit[lags - j : n_obs - j] for j in range(1, lags + 1)])
    xtx = x.T @ x
    if np.linalg.matrix_rank(xtx) < xtx.shape[0]:
        return np.nan

    beta = np.linalg.inv(xtx) @ x.T @ y
    resid = y - x @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return np.nan

    dq = float((beta.T @ xtx @ beta) / sigma2)
    return float(1.0 - chi2.cdf(dq, df=x.shape[1]))


def evaluate_var(df: pd.DataFrame, model: str, var_col: str, alpha=ALPHA) -> dict:
    real = df["loss_real"].to_numpy()
    var = df[var_col].to_numpy()
    _, _, hits = hit_series(real, var)
    n_obs = len(hits)
    n_exc = int(hits.sum())
    df_2020 = df.loc[df.index.year == 2020]

    return {
        "Model": model,
        "From": df.index.min().date().isoformat(),
        "To": df.index.max().date().isoformat(),
        "T": n_obs,
        "Expected Viol.": (1.0 - alpha) * n_obs,
        "Violations": n_exc,
        "Violation Rate": n_exc / n_obs,
        "VR Gap": abs(n_exc / n_obs - (1.0 - alpha)),
        "Coverage Bias": n_exc / n_obs - (1.0 - alpha),
        "Tick Loss": tick_loss(real, var, alpha=alpha),
        "Winkler Score": winkler_score(real, var, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var)),
        "Max VaR Level": float(np.nanmax(var)),
        "Min VaR Level": float(np.nanmin(var)),
        "Mean VaR 2020": float(df_2020[var_col].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020[var_col].max()) if len(df_2020) else np.nan,
        "n_negative_var": int(np.sum(var < 0)),
        "p_UC": kupiec_test(real, var, alpha=alpha),
        "p_CC": christoffersen_cc_test(real, var, alpha=alpha),
        "p_DQ": dq_test(real, var, alpha=alpha, lags=4),
    }


def add_test_labels(row: dict, sig=SIG) -> dict:
    row = dict(row)
    row["UC Test"] = "PASS" if row["p_UC"] > sig else "FAIL"
    row["CC Test"] = "PASS" if row["p_CC"] > sig else "FAIL"
    row["DQ Test"] = "PASS" if row["p_DQ"] > sig else "FAIL"
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return row


## Construccion de tablas finales

Esta celda lee las predicciones de cada modelo y prepara las tablas resumen, anuales y de violaciones.


In [ ]:
from pathlib import Path

import pandas as pd


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
PREDICTIONS = RESULTS / "predictions"
TABLES = RESULTS / "tables"
RESULTS.mkdir(exist_ok=True)
PREDICTIONS.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)


MODEL_FILES = [
    ("MLP-VaR", "mlp_var_predictions.csv", "VaR_pred", "mlp_var"),
    ("GARCH-t", "garch_t_var_predictions.csv", "VaR_GARCH", "garch_t"),
    ("GARCH-Normal", "garch_n_var_predictions.csv", "VaR_GARCH_n", "garch_normal"),
    ("Historical Simulation", "hs_var_predictions.csv", "VaR_HS", "historical_simulation"),
    ("CAViaR-AS", "caviar_var_predictions.csv", "VaR_CAViaR", "caviar_as"),
]


def load_model(filename: str, var_col: str) -> pd.DataFrame:
    df = pd.read_csv(PREDICTIONS / filename, index_col=0, parse_dates=True).sort_index()
    df = df.loc["2015-01-02":"2024-12-27"].copy()
    if "viol" not in df.columns:
        df["viol"] = (df["loss_real"] > df[var_col]).astype(int)
    if "year" not in df.columns:
        df["year"] = df.index.year
    return df


def main() -> None:
    rows = []
    yearly_rows = []
    for model, filename, var_col, key in MODEL_FILES:
        df = load_model(filename, var_col)
        summary_row = add_test_labels(evaluate_var(df, model=model, var_col=var_col))
        rows.append(summary_row)
        pd.DataFrame([summary_row]).to_csv(TABLES / f"{key}_summary_2015_2024.csv", index=False)

        model_yearly_rows = []
        for year, group in df.groupby(df.index.year):
            row = add_test_labels(evaluate_var(group, model=model, var_col=var_col))
            row["year"] = int(year)
            yearly_rows.append(row)
            model_yearly_rows.append(row)
        pd.DataFrame(model_yearly_rows).to_csv(TABLES / f"{key}_yearly_2015_2024.csv", index=False)

        violations = df.loc[df["viol"] == 1, [var_col, "loss_real", "viol", "year"]].copy()
        violations["exceedance"] = violations["loss_real"] - violations[var_col]
        violations["exceedance_ratio"] = violations["loss_real"] / violations[var_col]
        violations.to_csv(TABLES / f"{key}_violations_2015_2024.csv")

    summary = pd.DataFrame(rows)
    yearly = pd.DataFrame(yearly_rows)

    summary.to_csv(TABLES / "model_comparison_2015_2024.csv", index=False)
    yearly.to_csv(TABLES / "model_yearly_comparison_2015_2024.csv", index=False)

    print("Generated official result tables in", TABLES)


## Generacion de resultados oficiales

Esta celda guarda todas las tablas finales dentro de `results/tables/`.


In [ ]:
main()


## Revision de tablas

Esta celda muestra las tablas principales para comprobar visualmente los resultados finales.


In [ ]:
summary = pd.read_csv(TABLES / "model_comparison_2015_2024.csv")
yearly = pd.read_csv(TABLES / "model_yearly_comparison_2015_2024.csv")
display(summary)
display(yearly)
